# Inventaa GraphRAG - MCP Retrieval Test Notebook

Interactive test harness for the Tri-Store GraphRAG MCP server (mcp_server.py).
Assumes the server is already running:

```powershell
python mcp_server.py --transport streamable-http --port 8008
```

Test set focuses on the two primary paths: **find_product / browse_category**
(search_catalog) and **faq_knowledge**. `get_advice`, `check_policy`, and
`get_product_info` are intentionally omitted - the engine collapses them into
FIND_PRODUCT (graphrag_engine.py:97-105), so they add no distinct coverage.

> Cells use **top-level `await`** (Jupyter already runs an event loop, so
> `asyncio.run()` would raise `RuntimeError: asyncio.run() cannot be called from
> a running event loop`).
>
> Known issues discovered while building this notebook are flagged inline and
> summarized in the final cell.


## 0. Setup

In [16]:
import os, json
from fastmcp import Client

MCP_URL = os.getenv("MCP_URL", "http://127.0.0.1:8008/mcp")
# Canonical tenant id. NOTE: tenant_inventaa_led_001 in the doc is an ALIAS the MCP
# engine does NOT normalise (only router_api.py does) and graph_search.py filters
# WHERE p.tenant = $tenant_id, so the alias returns 0 graph hits. Use inventaa.
TENANT = os.getenv("TENANT", "inventaa")

print("MCP_URL =", MCP_URL, "| TENANT =", TENANT)


MCP_URL = http://127.0.0.1:8008/mcp | TENANT = inventaa


## 1. Discover the real server surface

The doc lists `get_catalog_status` as a tool and a `verify_whatsapp_status` tool.
Reality (per `mcp_server.py`): only `search_catalog`, `get_taxonomy_context`,
`get_product_details` are tools, and `get_catalog_status` is a RESOURCE
(`data://catalog/status`). `verify_whatsapp_status` does NOT exist.


In [17]:
async def discover():
    async with Client(MCP_URL) as c:
        tools = await c.list_tools()
        resources = await c.list_resources()
        return sorted(t.name for t in tools), sorted(str(r.uri) for r in resources)

tools, resources = await discover()
print("TOOLS:    ", tools)
print("RESOURCES:", resources)
assert "verify_whatsapp_status" not in tools, "verify_whatsapp_status should NOT exist"
assert "get_catalog_status" not in tools, "get_catalog_status is a RESOURCE, not a tool"
assert "data://catalog/status" in resources, "catalog status resource missing"


TOOLS:     ['get_product_details', 'get_taxonomy_context', 'search_catalog']
RESOURCES: ['data://catalog/status']


## 2. Helpers

In [18]:
async def call_tool(name, args):
    async with Client(MCP_URL) as c:
        res = await c.call_tool(name, args)
        data = getattr(res, "data", None)
        if isinstance(data, dict):
            return data
        return json.loads(res.content[0].text) if res.content else None

async def read_resource(uri):
    async with Client(MCP_URL) as c:
        content = await c.read_resource(uri)
        if isinstance(content, list):
            return [getattr(x, "text", str(x)) for x in content]
        return getattr(content, "text", str(content))

def base_intent(intent):
    return {"intent": intent, "category_keywords": [], "feature_keywords": [], "product_name": None, "filters": {}, "preferences": {}}


## 3A. `search_catalog` - intent `faq_knowledge`

**OBSERVATION (BUG):** the live server returns `intent: check_policy` for a
`faq_knowledge` request. `src/query/graphrag_engine.py:95-99` should map it to
`FAQ_KNOWLEDGE`. The running server is on a different/older build - restart it.


In [19]:
args = {"query": "lakeside resort project",
    "limit": 5, "tenant_id": TENANT, "session_id": "test_session_knowledge_01",
    "intent_data": {**base_intent("faq_knowledge"), "category_keywords": []}}
r = await call_tool("search_catalog", args)
print(json.dumps(r, indent=2, default=str)[:2000])
print("\n>>> returned intent:", r.get("intent"), "(expected faq_knowledge)")
chunks = r.get("chunks", [])
print(">>> chunks:", len(chunks), "| context_text present:", bool(r.get("context_text")))
for i, c_ in enumerate(chunks, 1):
    print(f"  [{i}] {c_[:4000]}")


{
  "status": "success",
  "intent": "faq",
  "items": [],
  "item_links": [],
  "chunks": [],
  "context_text": "No specific matching items found in the catalog.",
  "response": "No matching products or knowledge entries found."
}

>>> returned intent: faq (expected faq_knowledge)
>>> chunks: 0 | context_text present: True


## 3B. `search_catalog` - intent `find_product` with price filter

In [10]:
args = {"query": "Show me waterproof LED gate pillar lights under 1500 rupees",
    "limit": 6, "tenant_id": TENANT, "session_id": "test_session_find_01",
    "intent_data": {"intent": "find_product", "category_keywords": ["Gate Light Collections"],
        "feature_keywords": ["waterproof", "IP65-rated"], "product_name": None,
        "filters": {"category": "Gate Light Collections"}, "preferences": {"max_price": 1500}}}
r = await call_tool("search_catalog", args)
print(json.dumps(r, indent=2, default=str)[:2000])
prods = r.get("items", [])
over = [p["sku"] for p in prods if p.get("price_num", 0) > 1500]
print("\n>>> returned intent:", r.get("intent"))
print(">>> products over 1500:", over or "none")


{
  "status": "success",
  "intent": "find_product",
  "products": [],
  "product_links": [],
  "chunks": [
    "[Context: BEST SELLERS]\n\\\\\n**Bloom Solar LED Outdoor Gate Light**\\\\\n\\\\\n\u2605\u2605\u2605\u2605\u26054.9(44)\\\\\n\\\\\n\u20b93,709\u20b95,45532% OFF\\\\\n\\\\\n1 color available](https://inventaa.in/products/bloom-led-solar-gate-light)  \n[TRENDING \u26a1![Lexa Lens Pathway Street Light](https://cdn.shopify.com/s/files/1/0812/4930/4891/files/FrontImage24W.jpg?v=1757412524)\\\\\n\\\\\n**Lexa Lens LED Pathway Street Light**\\\\\n\\\\\n\u2605\u2605\u2605\u2605\u26055.0(29)\\\\\n\\\\\n\u20b9987\u20b91,45132% OFF\\\\\n\\\\\n1 color available](https://inventaa.in/products/lexa-led-street-light?_pos=1&_psq=Lexa+Lens+LED+Pathway+Street+Light&_ss=e&_v=1.0)  \n[BEST VALUE \ud83d\udc8e![Viva 12W Gate Pillar Light](https://cdn.shopify.com/s/files/1/0812/4930/4891/files/VivaLHGoldCool.jpg?v=1757412464)\\\\\n\\\\\n**Viva Gate Pillar Lights 12W LED - Modern Exterior Door Light**

## 3C. `search_catalog` - intent `browse_category`

In [24]:
args = {"query": "Show me what outdoor wall lights collections you have",
    "limit": 10, "tenant_id": TENANT, "session_id": "test_session_browse_01",
    "intent_data": {"intent": "browse_category", "category_keywords": ["Outdoor Wall Lamps"],
        "feature_keywords": [], "product_name": None, "filters": {"category": "Outdoor Wall Lamps"}, "preferences": {}}}
r = await call_tool("search_catalog", args)
print(json.dumps(r, indent=2, default=str)[:4000])
print("\n>>> returned intent:", r.get("intent"), "| #products:", len(r.get("items", [])))


{
  "status": "success",
  "intent": "browse",
  "items": [
    {
      "sku": "2Y-W133",
      "name": "Ayla LED Outdoor Wall Mount Light",
      "price_num": 1780,
      "regular_price": "2374.16",
      "discount_percentage": 25,
      "rating_score": 5.0,
      "review_count": 56,
      "url": "https://inventaa.in/products/ayla-led-outdoor-wall-light-for-home-elevation-with-1-year-replacement-warranty",
      "image_url": "http://inventaa.in/cdn/shop/files/Ayla_2W_2_Way_LED_Wall_Light.jpg?v=1757412436&width=2048",
      "description": "Elevate the aesthetic and functionality of your home's exterior with the Ayla LED Outdoor Wall Light\nDesigned to blend seamlessly with any architectural style, this sophisticated lighting solution not only enhances the beauty of your property but also ensures safety and security with its superior illumination\nFeatures Modern Design: Sleek and contemporary, the Ayla LED Outdoor Wall Light complements a variety of home designs, from traditional to mo

## 4. `get_product_details` - resolve a REAL SKU first

The doc hardcodes SKU `12M-2026B` which may not exist (real: `18C-2042`, `MAR03C`, `GAT05`).
We discover a real SKU via a browse search, then look it up.


In [29]:
async def find_a_sku():
    args = {"query": "show outdoor  lights", "limit": 3, "tenant_id": TENANT, "session_id": "test_sku_discovery",
        "intent_data": {"intent": "browse_category", "category_keywords": ["Outdoor lights"], "feature_keywords": [], "product_name": None,
            "filters": {"category": "Outdoor lights"}, "preferences": {}}}
    r = await call_tool("search_catalog", args)
    print(json.dumps(r, indent=2, default=str)[:2000])
    links = r.get("product_links") or []
    if links: return links[0]["sku"]
    prods = r.get("items") or []
    return prods[0]["sku"] if prods else None

REAL_SKU = await find_a_sku()
print("Discovered real SKU:", REAL_SKU)
if REAL_SKU:
    det = await call_tool("get_product_details", {"sku": REAL_SKU, "tenant_id": TENANT})
    print(json.dumps(det, indent=2, default=str)[:2000])
else:
    print("No SKU discovered - browse_category returned 0 products")


{
  "status": "success",
  "intent": "browse",
  "items": [
    {
      "sku": "6C-W32",
      "name": "Reina Up Down Elevation LED Lights",
      "price_num": 800,
      "regular_price": "1067.9",
      "discount_percentage": 25,
      "rating_score": 5.0,
      "review_count": 56,
      "url": "https://inventaa.in/products/reina-wall-light",
      "image_url": "http://inventaa.in/cdn/shop/files/6w_f96448a8-5ada-4ae3-a2e8-5b39dcd8b80c.jpg?v=1766036112&width=2048",
      "description": "This Reina LED Outdoor Up And Down Light is your one-stop solution to elevate your exterior spaces with a perfect blend of style and functionality\nIts sleek, aluminum body not only offers durability but also a modern aesthetic that complements any outdoor fixture\nThis unbreakable design guarantees a long-lasting companion, standing strong against harsh weather elements\nThis up and down elevation light radiates 6/12 watts of warm light, effortlessly casting a welcoming glow that transforms any outdoor

## 3D. Product-name retrieval (canonical flow) + bare-query regression guard

**Canonical flow:** the WhatsApp agent extracts a product name from the user
query (via taxonomy candidates -> LLM) and passes it as `product_name` (string).
Our tool then runs a Lucene/fulltext search (`graph_search` ->
`db.index.fulltext.queryNodes` on `product_name_ft`) and hydrates the match(es)
from SQLite. 3D.1 locks that in.

**Regression guard:** a bare product-name query sent *without* an extracted
`product_name` previously returned a broad-navigation greeting and 0 products
because the engine's `is_broad_query` guard swallowed it before retrieval ran.
3D.2 ensures it still falls through to a real search. 3D.3 confirms genuine
browse requests still resolve to the category menu.


In [33]:
# 3D. Product-name retrieval (canonical flow) + bare-query regression guard.
#
# CANONICAL FLOW: agent passes product_name (string) -> Lucene/fulltext search on
# product_name_ft (Neo4j) + SQLite ILIKE -> hydrated product(s).
def _is_broad_nav_greeting(r):
    # broad-nav response: 0 products and a greeting in context_text/response
    return len(r.get("items", [])) == 0 and bool(r.get("context_text") or r.get("response"))

# 3D.1 agent passes product_name -> Lucene/fulltext search must return the product
args = {"query": "athena", "limit": 5, "tenant_id": TENANT,
        "session_id": "test_session_prodname_01",
        "intent_data": {"intent": "find_product", "category_keywords": [],
                          "feature_keywords": [], "product_name": "athena",
                          "filters": {}, "preferences": {}}}
r = await call_tool("search_catalog", args)
prods = r.get("items", [])
print(">>> [3D.1] intent:", r.get("intent"), "| #products:", len(prods))
assert len(prods) > 0, "product_name='athena' returned 0 products (Lucene/fulltext path broken)"
assert any("athena" in p["name"].lower() for p in prods), "No product name contains 'athena'"
print(">>> matches:", [p["name"] for p in prods])
print("[OK] 3D.1 agent-provided product_name -> Lucene search returns the product")

# 3D.2 defensive guard: bare product-name query WITHOUT product_name must NOT be
# swallowed by broad navigation (engine must fall through to retrieval).
args2 = {"query": "athena", "limit": 5, "tenant_id": TENANT,
         "session_id": "test_session_bare_name_01",
         "intent_data": {**base_intent("find_product"), "product_name": None}}
r2 = await call_tool("search_catalog", args2)
prods2 = r2.get("items", [])
print("\n>>> [3D.2] bare 'athena' (no product_name) #products:", len(prods2),
      "| broad-nav?", _is_broad_nav_greeting(r2))
assert len(prods2) > 0, "REGRESSION: bare product-name 'athena' (no product_name) returned 0 products"
assert not _is_broad_nav_greeting(r2), "REGRESSION: bare 'athena' returned a broad-nav greeting"
print("[OK] 3D.2 bare product-name query is not swallowed by broad navigation")

# 3D.3 sanity: a generic browse request must STILL resolve to broad navigation.
args3 = {"query": "show me your collections", "limit": 5, "tenant_id": TENANT,
         "session_id": "test_session_bare_name_02",
         "intent_data": {**base_intent("find_product"), "product_name": None}}
r3 = await call_tool("search_catalog", args3)
print("\n>>> [3D.3] 'show me your collections' broad-nav?", _is_broad_nav_greeting(r3))
assert _is_broad_nav_greeting(r3), "Broad-nav request was NOT treated as navigation"
print("[OK] 3D.3 broad-nav requests still resolve to the category menu")


>>> [3D.1] intent: find_item | #products: 2
>>> matches: ['Athena 18W LED Classic Design - Outdoor Gate Light', 'Athena 3 in 1 Gate Post Lights -  Frontgate Lighting']
[OK] 3D.1 agent-provided product_name -> Lucene search returns the product

>>> [3D.2] bare 'athena' (no product_name) #products: 2 | broad-nav? False
[OK] 3D.2 bare product-name query is not swallowed by broad navigation

>>> [3D.3] 'show me your collections' broad-nav? True
[OK] 3D.3 broad-nav requests still resolve to the category menu


## 5. `get_taxonomy_context`

In [10]:
r = await call_tool("get_taxonomy_context", {"query": "waterproof solar gate lamps", "threshold": 0.80})
print(json.dumps(r, indent=2, default=str)[:2000])


{
  "status": "success",
  "taxonomy": {
    "category": [
      "Solar"
    ],
    "feature": [
      "waterproof"
    ],
    "use_case": [
      "gate-pillar",
      "solar-outdoor"
    ]
  }
}


## 6. `get_catalog_status` - RESOURCE `data://catalog/status`

The doc calls this as a tool; it is actually a resource. Reading it as a tool fails.


In [11]:
r = await read_resource("data://catalog/status")
print(r[0] if isinstance(r, list) else r)


{"status": "healthy", "sqlite_connected": true, "total_products_in_catalog": 125, "neo4j_lucene_indexes": [{"name": "chunk_text", "state": "ONLINE"}, {"name": "product_name_ft", "state": "ONLINE"}]}


## 7. `verify_whatsapp_status` - DOES NOT EXIST

The doc lists this as a tool but `mcp_server.py` exposes no such tool. This cell
demonstrates the failure so it is documented, not silently ignored.


In [12]:
try:
    r = await call_tool("verify_whatsapp_status", {"tenant_id": TENANT})
    print("UNEXPECTED success:", r)
except Exception as e:
    print("EXPECTED FAILURE ->", type(e).__name__, str(e)[:200])


EXPECTED FAILURE -> ToolError Unknown tool: 'verify_whatsapp_status'


## 9. Candidate-selection approaches (re-prioritized test plan)

Test harness for the five candidate-selection approaches we prioritized over the
GraphRAG survey's KG-reasoning branch (cross-encoder rerank + scope_skus, history-
grounded dense product ANN, structured NLU slot-filling, pick-feedback loop, RAPTOR
for FAQ/policy). Implemented paths (slot-filling, FAQ) have real assertions; the rest
are scaffolded as TODO so the notebook stays executable end-to-end. Flip them to
assertions once the server capability lands.


In [ ]:
# 9.1 Cross-encoder reranker over hybrid recall + scope_skus
#
# RECALL stage (exists): broad hybrid retrieval returns a candidate set.
# RERANK stage (TODO): a cross-encoder reranks those candidates using the grounded
# (history + utterance) query. scope_skus (TODO) constrains recall to the products
# already shown, so comparative follow-ups ("the cheaper one") stay within the shown
# set instead of re-querying the whole catalog.
args = {"query": "gate lights under 1500", "limit": 8, "tenant_id": TENANT,
        "session_id": "test_recall_01",
        "intent_data": {**base_intent("find_product"),
                        "filters": {"category": "Gate Light Collections"},
                        "preferences": {"max_price": 1500}}}
r = await call_tool("search_catalog", args)
recall = r.get("items", [])
print(">>> [9.1] recall candidates:", len(recall))
for p in recall[:5]:
    print("   -", p["sku"], p["name"], "Rs.", p.get("price_num"))
# scope_skus is not implemented on the server yet; probe and report (do not hard-fail).
probe = dict(args); probe["scope_skus"] = [p["sku"] for p in recall[:3]]
try:
    r2 = await call_tool("search_catalog", probe)
    scoped = r2.get("items", [])
    print(">>> scope_skus probe returned:", len(scoped), "(if unconstrained, server ignored the param)")
    print("[NOTE] Enable assertion len(scoped) <= len(recall) and all in scope once scope_skus is wired in.")
except Exception as e:
    print(">>> [TODO] scope_skus / reranker not implemented yet:", type(e).__name__, str(e)[:120])


In [ ]:
# 9.2 History-grounded dense product ANN (per-tenant embeddings)
#
# TODO: index product embeddings per tenant; embed the GROUNDED query (history +
# current utterance) and run ANN. Natively handles follow-ups ("the cheaper one")
# because the embedding sees conversation context, not just the bare utterance.
# Same idea as Inventaa's cosine retriever but with history + multi-tenant indexes.
# No product-ANN index exists on the server yet (vector_search covers :Chunk/:FAQ only).
print(">>> [9.2] Product ANN index: NOT IMPLEMENTED")
print("    Test plan: embed grounded_query -> ANN over tenant product index ->")
print("    assert the focused product from history ranks top-1 for a follow-up.")


In [ ]:
# 9.3 Structured NLU slot-filling instead of the taxonomy call
#
# The agent extracts slots (product_name / category / feature / filters) from
# (history + utterance) and passes them as intent_data. Retrieval is then driven by
# SLOTS, not by a taxonomy keyword match on the raw text. Taxonomy becomes a
# validator only. This cell proves a query with NO taxonomy signal still retrieves
# correctly when the product_name slot is populated (the agent's primary path).
args = {"query": "show me that one", "limit": 5, "tenant_id": TENANT,
        "session_id": "test_slots_01",
        "intent_data": {"intent": "find_product", "category_keywords": [],
                        "feature_keywords": [], "product_name": "Athena 18W",
                        "filters": {}, "preferences": {}}}
r = await call_tool("search_catalog", args)
prods = r.get("items", [])
print(">>> [9.3] slot-driven retrieval #products:", len(prods))
assert len(prods) > 0, "Slot-filling path returned 0 products"
assert any("athena" in p["name"].lower() for p in prods), "product_name slot not honored"
print("[OK] 9.3 product_name slot drives retrieval without any taxonomy call")


In [ ]:
# 9.4 Pick-feedback loop (commerce signal)
#
# TODO: when the user picks a product from candidates, log the pick (session, sku,
# query) to a per-tenant feedback store. Future recall/rerank uses it as a signal
# (boost picked / similar items). Not implemented; this cell sketches the test.
print(">>> [9.4] Pick-feedback store: NOT IMPLEMENTED")
print("    Test plan:")
print('      1. recall "gate lights" -> candidates [A, B, C]')
print("      2. simulate user pick -> log_feedback(session, sku=B)")
print('      3. follow-up "show me similar" -> assert B (or siblings) ranked higher')
print("    Highest-leverage commerce signal; absent from the GraphRAG survey framing.")


In [31]:
# 9.5 RAPTOR-style hierarchical retrieval for FAQ / policy (not product search)
#
# The faq_knowledge channel already returns (:Chunk / :FAQ) context. RAPTOR would add
# a hierarchical summary index over those chunks so broad policy questions hit a
# summary node first, then drill down. Here we assert the FAQ channel works today.
args = {"query": "what is the warranty policy", "limit": 5, "tenant_id": TENANT,
        "session_id": "test_faq_01",
        "intent_data": {**base_intent("faq_knowledge")}}
r = await call_tool("search_catalog", args)
chunks = r.get("chunks", [])
print(">>> [9.5] faq_knowledge chunks:", len(chunks), "| intent:", r.get("intent"))
assert r.get("intent") == "faq_knowledge", "faq_knowledge intent not preserved"
assert len(chunks) > 0, "FAQ/policy channel returned no chunks"
print("[OK] 9.5 FAQ/policy retrieval works; RAPTOR hierarchy is an enhancement layer")


>>> [9.5] faq_knowledge chunks: 0 | intent: faq


AssertionError: faq_knowledge intent not preserved

## 8. Summary of doc-vs-code misalignments

| Doc section | Problem | Fix |
|---|---|---|
| 4 get_catalog_status | Called as a tool; it is a RESOURCE data://catalog/status | Read via read_resource |
| 5 verify_whatsapp_status | Tool does not exist in mcp_server.py | Remove from doc or implement |
| All tenant_id | Doc uses alias tenant_inventaa_led_001; MCP path does not normalise it, graph channel returns 0 hits | Use canonical inventaa |
| get_product_info / get_advice / check_policy | No dedicated engine branch; collapse to FIND_PRODUCT (intentionally omitted from tests) | Document or add handling |
| 2 SKU 12M-2026B | May not exist in catalog | Use a confirmed SKU from the DB |
| A faq_knowledge | Live server returns check_policy though code maps to FAQ_KNOWLEDGE (graphrag_engine.py:95-99) | Restart the server - it is running a stale build |
